Directional topological descriptor comparison

In [ ]:
import numpy as np
import pandas as pd
import gudhi as gd
from utils import LowerStarLayer, ecg_comparison
from tqdm.auto import tqdm
import wfdb

def load_mitbih_beats(record_id='100', window=200):
    record = wfdb.rdrecord(f'mitbih_data/{record_id}')
    annotation = wfdb.rdann(f'mitbih_data/{record_id}', 'atr')
    signal = record.p_signal[:, 0]
    X = []
    y = []
    for i, label in enumerate(annotation.symbol):
        if label not in ['N', 'V']:
            continue
        idx = annotation.sample[i]
        if idx - window < 0 or idx + window > len(signal):
            continue
        beat = signal[idx - window:idx + window]
        X.append(beat)
        y.append(0 if label == 'N' else 1)
    return (np.array(X), np.array(y))

def load_mitbih_all(records_id, window=200, classes=('N', 'V')):
    """
    Loads all MIT-BIH records into a single dataset.

    Returns:
        X: (n_beats, 2*window)
        y: (n_beats,)
    """
    X_all = []
    y_all = []
    for rec in tqdm(records_id):
        try:
            record = wfdb.rdrecord(f'mitbih_data/{rec}')
            ann = wfdb.rdann(f'mitbih_data/{rec}', 'atr')
            signal = record.p_signal[:, 0]
            for i, label in enumerate(ann.symbol):
                if label not in classes:
                    continue
                idx = ann.sample[i]
                if idx - window < 0 or idx + window >= len(signal):
                    continue
                beat = signal[idx - window:idx + window]
                X_all.append(beat)
                y_all.append(0 if label == classes[0] else 1)
        except Exception as e:
            print(f'Skipping record {rec}: {e}')
    return (np.array(X_all), np.array(y_all))

def balance_dataset(X, y):
    idx_n = np.where(y == 0)[0]
    idx_v = np.where(y == 1)[0]
    idx_n = np.random.choice(idx_n, size=len(idx_v), replace=False)
    idx = np.concatenate([idx_n, idx_v])
    return (X[idx], y[idx])

def build_lowerstar_layer(n_points):
    stbase = gd.SimplexTree()
    for i in range(n_points - 1):
        stbase.insert([i, i + 1], -10000000000.0)
    layer = LowerStarLayer(simplextree=stbase)
    return layer

np.random.seed(42)
records = ['100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '111', '112', '113', '114', '115', '116', '117', '118', '119', '121', '122', '123', '124', '200', '201', '202', '203', '205', '207', '208', '209', '210', '212', '213', '214']
X, y = load_mitbih_all(records_id=records)
print(X.shape)
print(y.shape)
unique, counts = np.unique(y, return_counts=True)
dict(zip(unique, counts))
X, y = balance_dataset(X, y)
print(X.shape)
print(y.shape)
unique, counts = np.unique(y, return_counts=True)
dict(zip(unique, counts))

n_points = 400
layer = build_lowerstar_layer(n_points)
results, timing, folds = ecg_comparison(X, y, layer, n_points, output_prefix="mitbih")

display(results)
display(timing)

 71%|███████▏  | 25/35 [00:02<00:00, 11.88it/s]